# Mastering Python `__format__`: Advanced Problems and Complete Solutions

**Audience:** intermediate-to-advanced Python developers. **Runtime:** Python 3.10+; only the standard library is needed. **Structure:** conceptual reference, worked examples, 12 progressively harder exercises, fully implemented solutions, and executable regression tests. Execute cells **top to bottom**; all timestamps and test inputs are deterministic.

**Objectives:** distinguish `__str__`, `__repr__`, and `__format__`; design explicit format mini-languages; delegate numeric/date/string formatting; handle conversion flags, precision, alignment, negative values, Unicode, invalid specs, and output-size limits; test contracts without side effects.

> Each exercise states a contract *before* its solution. To practice, hide the following solution cell and implement the API yourself; run the supplied assertions afterward. Solutions deliberately reject unspecified options rather than silently guessing.

## 0. Reference: the formatting protocol

- `str(x)` calls `x.__str__()`; `repr(x)` calls `x.__repr__()`.
- `format(x, spec)` calls `type(x).__format__(x, spec)`; f-strings and `"{:spec}".format(x)` use the same underlying protocol. An omitted spec is `""`.
- `f"{x!s:spec}"` **first** applies `str(x)` and then formats the resulting *string*. `!r` and `!a` similarly convert before applying the *string's* format spec; these conversions bypass the original object's `__format__`.
- `__format__` **must** return a `str`. The inherited `object.__format__` accepts the empty spec (via `str(x)`) but rejects nonempty specs. A custom `__format__` is free to define different empty-spec semantics: document them.
- Built-in float, `Decimal`, `str`, and date/time objects each have their **own** spec grammar. Do not assume that `%Y-%m-%d` is valid for a number or that `,.2f` is valid for `datetime`.
- Avoid debug printing in `__repr__`, `__str__`, or `__format__`; formatting is often invoked inside logging, debugging, and display code.

**Best-practice contract:** define a small grammar, delegate to built-ins where possible, validate unsupported modes explicitly, keep conversion side-effect-free, and test both valid and invalid inputs.

In [1]:
from dataclasses import dataclass
from datetime import date, datetime, timedelta, timezone
from decimal import Decimal, ROUND_HALF_EVEN
from io import StringIO
import csv
import math
import re


def raises(expected, operation):
    """Tiny standard-library-only equivalent of an exception assertion."""
    try:
        operation()
    except expected:
        return
    except Exception as exc:
        raise AssertionError(
            f"Expected {expected.__name__}, got {type(exc).__name__}: {exc}"
        ) from exc
    raise AssertionError(f"Expected {expected.__name__} to be raised")

print("Imports and test helper ready.")

Imports and test helper ready.


## Worked example A — determine which method actually runs

`!s`, `!r`, and `!a` are *conversions*, not custom format modes. An alignment spec after a conversion belongs to `str`, not the original object. The trace below has no print side effects inside the methods.

In [2]:
class FormatSpy:
    def __init__(self):
        self.calls = []

    def __str__(self):
        self.calls.append("str")
        return "S"

    def __repr__(self):
        self.calls.append("repr")
        return "R"

    def __format__(self, spec):
        self.calls.append(("format", spec))
        return f"F[{spec}]"


spy = FormatSpy()
assert format(spy) == "F[]"
assert f"{spy:wide}" == "F[wide]"
assert f"{spy!s:>4}" == "   S"
assert f"{spy!r:<3}" == "R  "
assert spy.calls == [
    ("format", ""), ("format", "wide"), "str", "repr"
]
print("Dispatch:", spy.calls)
print("Decimal delegated:", format(Decimal("1234.5"), ">12,.2f"))
print("Date delegated:", format(date(2024, 2, 29), "%d %b %Y"))

Dispatch: [('format', ''), ('format', 'wide'), 'str', 'repr']
Decimal delegated:     1,234.50
Date delegated: 29 Feb 2024


## Worked example B — built-in behavior and return-type contract

Do not manually call an object's `__format__` in application code; use `format(obj, spec)` or an f-string. Explicit checks demonstrate what happens if a class inherits the default behavior or violates the return-type contract.

In [3]:
class Plain:
    def __str__(self):
        return "plain"


class Broken:
    def __format__(self, spec):
        return 42  # Incorrect: never do this.


assert format(Plain()) == "plain"
raises(TypeError, lambda: format(Plain(), ">10"))
raises(TypeError, lambda: format(Broken(), ""))
assert format(0.1, ".20f") == "0.10000000000000000555"
print("Object fallback, invalid return values, and float precision verified.")

Object fallback, invalid return values, and float precision verified.


## Worked example C — using nested replacement fields

The dynamic width and precision are resolved *before* calling the formatted object's `__format__`. Thus `f"{value:{width}.{precision}f}"` sends a fully formed string such as `"10.2f"` to the value.

In [4]:
value = Decimal("12.3456")
width, precision = 10, 2
spec = f"{width}.{precision}f"
assert spec == "10.2f"
assert f"{value:{width}.{precision}f}" == format(value, spec)
assert format(date(2025, 1, 2), "%Y/%m/%d") == "2025/01/02"
print("Resolved format spec:", repr(spec), "->", repr(format(value, spec)))

Resolved format spec: '10.2f' -> '     12.35'


# Problem 1 — `Person`: clean representations and delegated date specs

**Difficulty:** intermediate. Build an immutable `Person(name, dob)` with:

1. `str(person)` → `"Person(Alex)"`.
2. `repr(person)` → a useful, explicit string with quoted name and ISO date (no `print` calls).
3. `format(person, "")` → the same string as `str(person)`.
4. `format(person, "%B %d, %Y")` → `"Alex — October 20, 1900"` (use `date.__format__`, not a handwritten month lookup).

**Edge cases:** leap days, Unicode names, malformed date directives. Date formatting behavior for directives is platform-dependent, so tests use portable directives only.

In [5]:
# SOLUTION 1
@dataclass(frozen=True)
class Person:
    name: str
    dob: date

    def __post_init__(self):
        if not isinstance(self.name, str) or not self.name:
            raise ValueError("name must be a nonempty string")
        if not isinstance(self.dob, date) or isinstance(self.dob, datetime):
            raise TypeError("dob must be a date, not datetime")

    def __str__(self):
        return f"Person({self.name})"

    def __repr__(self):
        return f"Person(name={self.name!r}, dob={self.dob.isoformat()!r})"

    def __format__(self, spec):
        if not spec:
            return str(self)
        return f"{self.name} — {format(self.dob, spec)}"


alex = Person("Alex", date(1900, 10, 20))
assert str(alex) == "Person(Alex)"
assert repr(alex) == "Person(name='Alex', dob='1900-10-20')"
assert format(alex) == str(alex)
assert format(alex, "%B %d, %Y") == "Alex — October 20, 1900"
assert f"{alex!s:>16}" == format(str(alex), ">16")
assert format(Person("Zoë", date(2000, 2, 29)), "%Y-%m-%d") == "Zoë — 2000-02-29"
raises(ValueError, lambda: Person("", date(2000, 1, 1)))
print("Problem 1: PASS")

Problem 1: PASS


# Problem 2 — `Money`: a small, documented formatting language

**Difficulty:** advanced. Represent money with `Decimal`, not binary floats. Support the grammar **`MODE|NUMERIC_SPEC`**, where the mode is `symbol`, `code`, or `account`. A missing mode/spec (`""`) means `symbol|,.2f`; a mode with no `|` also uses `,.2f`. Use USD `$`, EUR `€`, and GBP `£`; reject unknown currencies. Quantize to cents using half-even rounding at construction and reject non-finite amounts. `account` renders negatives in parentheses; other modes retain the minus sign. Delegate `NUMERIC_SPEC` to `Decimal.__format__`.

Examples: `$1,234.50`, `USD 1,234.50`, `($1,234.50)`. **Design choice:** numeric specs control the *magnitude only*; they should not be used to set the sign, and should be limited to your application's documented choices if exposed to untrusted users (Problem 10).

In [6]:
# SOLUTION 2
_CURRENCY_SYMBOLS = {"USD": "$", "EUR": "€", "GBP": "£"}


@dataclass(frozen=True)
class Money:
    amount: Decimal
    currency: str = "USD"

    def __post_init__(self):
        if self.currency not in _CURRENCY_SYMBOLS:
            raise ValueError(f"unsupported currency: {self.currency!r}")
        if not isinstance(self.amount, Decimal):
            raise TypeError("amount must be Decimal; construct with Decimal('1.23')")
        if not self.amount.is_finite():
            raise ValueError("amount must be finite")
        object.__setattr__(
            self, "amount", self.amount.quantize(Decimal("0.01"), rounding=ROUND_HALF_EVEN)
        )

    def __str__(self):
        return self.__format__("")

    def __format__(self, spec):
        mode, separator, numeric_spec = spec.partition("|")
        if not mode:
            mode = "symbol"
        if mode not in {"symbol", "code", "account"}:
            raise ValueError(f"unsupported money mode: {mode!r}")
        if not separator or not numeric_spec:
            numeric_spec = ",.2f"
        magnitude = format(abs(self.amount), numeric_spec)
        symbol = _CURRENCY_SYMBOLS[self.currency]
        body = (f"{self.currency} {magnitude}" if mode == "code"
                else f"{symbol}{magnitude}")
        if self.amount < 0:
            return f"({body})" if mode == "account" else f"-{body}"
        return body


positive = Money(Decimal("1234.505"))
negative = Money(Decimal("-1234.505"))
assert positive.amount == Decimal("1234.50")  # half-even tie
assert format(positive) == "$1,234.50"
assert format(positive, "code") == "USD 1,234.50"
assert format(negative, "account") == "($1,234.50)"
assert format(Money(Decimal("1.2"), "EUR"), "symbol|.3f") == "€1.200"
assert format(negative, "symbol|,.2f") == "-$1,234.50"
assert f"{positive:code|,.2f}" == "USD 1,234.50"
raises(ValueError, lambda: Money(Decimal("NaN")))
raises(TypeError, lambda: Money(1.2))
raises(ValueError, lambda: format(positive, "bitcoin"))
raises(ValueError, lambda: Money(Decimal("1"), "XYZ"))
print("Problem 2: PASS")

Problem 2: PASS


# Problem 3 — string transformations plus native alignment

**Difficulty:** advanced. Build `Label(text)` with grammar `MODE|STRING_SPEC`, where `MODE` is `raw`, `upper`, or `lower`. An omitted spec defaults to `raw` with no alignment. When supplied, `STRING_SPEC` must be delegated to `str.__format__`. Reject unsupported modes. Show why formatting a string *after* applying `!r` is not the same as invoking the label's custom formatter.

Examples: `format(Label("Cat"), "upper|^9") == "   CAT   "`; `f"{label!r:>20}"` formats the `repr` string instead of entering the label grammar.

In [7]:
# SOLUTION 3
@dataclass(frozen=True)
class Label:
    text: str

    def __format__(self, spec):
        if not spec:
            return self.text
        mode, sep, text_spec = spec.partition("|")
        if mode not in {"raw", "upper", "lower"}:
            raise ValueError(f"unknown label mode: {mode!r}")
        transformed = {
            "raw": lambda x: x,
            "upper": str.upper,
            "lower": str.lower,
        }[mode](self.text)
        return format(transformed, text_spec if sep else "")


label = Label("Cat")
assert format(label) == "Cat"
assert format(label, "upper|^9") == "   CAT   "
assert format(label, "lower|*>6") == "***cat"
assert format(label, "raw|.2") == "Ca"
assert f"{label!r:>20}" == format(repr(label), ">20")
raises(ValueError, lambda: format(label, "title|^8"))
print("Problem 3: PASS")

Problem 3: PASS


# Problem 4 — `Duration`: sign-safe arithmetic without floats

**Difficulty:** advanced. Build an immutable `Duration(delta: timedelta)` supporting `clock`, `compact`, and `iso` (empty spec defaults to `clock`). `clock` uses **total hours**, not hours modulo 24 (`27:04:05`); `compact` uses day/hour/minute/second units (`1d 3h 4m 5s`); `iso` follows a restricted ISO-8601 duration form `P[nD]T[nH][nM][nS]` with zero represented as `PT0S`. For negative durations, prefix the *entire string* with `-`. Support only exact-second durations; raise `ValueError` for microseconds. Use integer arithmetic on `timedelta` components, not `total_seconds()` floats.

Boundary cases: zero, negative values, 25+ hours, and a one-microsecond input.

In [8]:
# SOLUTION 4
@dataclass(frozen=True)
class Duration:
    delta: timedelta

    def __format__(self, spec):
        mode = spec or "clock"
        if mode not in {"clock", "compact", "iso"}:
            raise ValueError(f"unknown duration mode: {mode!r}")
        if self.delta.microseconds:
            raise ValueError("duration must contain whole seconds")
        total = self.delta.days * 86_400 + self.delta.seconds
        sign = "-" if total < 0 else ""
        absolute = abs(total)
        days, remainder = divmod(absolute, 86_400)
        hours, remainder = divmod(remainder, 3_600)
        minutes, seconds = divmod(remainder, 60)
        if mode == "clock":
            full_hours, rem = divmod(absolute, 3_600)
            minute, second = divmod(rem, 60)
            return f"{sign}{full_hours:02d}:{minute:02d}:{second:02d}"
        if mode == "compact":
            parts = [f"{days}d"] if days else []
            parts.extend((f"{hours}h", f"{minutes}m", f"{seconds}s"))
            return sign + " ".join(parts)
        date_part = f"{days}D" if days else ""
        time_part = "".join(
            (f"{n}{unit}" if n else "")
            for n, unit in ((hours, "H"), (minutes, "M"), (seconds, "S"))
        ) or "0S"
        return f"{sign}P{date_part}T{time_part}"


long = Duration(timedelta(days=1, hours=3, minutes=4, seconds=5))
assert format(long) == "27:04:05"
assert format(long, "compact") == "1d 3h 4m 5s"
assert format(long, "iso") == "P1DT3H4M5S"
assert format(Duration(timedelta()), "iso") == "PT0S"
assert format(Duration(timedelta(seconds=-1)), "clock") == "-00:00:01"
assert format(Duration(timedelta(seconds=-1)), "iso") == "-PT1S"
raises(ValueError, lambda: format(long, "unknown"))
raises(ValueError, lambda: format(Duration(timedelta(microseconds=1)), "clock"))
print("Problem 4: PASS")

Problem 4: PASS


# Problem 5 — timezone-aware event formatting

**Difficulty:** advanced. `Event(title, when)` must accept **aware** `datetime` instances only (`when.utcoffset() is not None`). Format grammar: `local` (default: ISO-8601 in the object's timezone), `utc` (ISO-8601 normalized to UTC), or `date:STRFTIME_SPEC` (delegate the specified directive to the object's original timezone). Reject empty date directives and unknown modes. Never use `datetime.utcnow()` here: fixed aware fixtures are portable and reproducible.

**Think about it:** UTC normalization changes the wall-clock date near midnight; date formatting intentionally retains the event's original timezone.

In [9]:
# SOLUTION 5
@dataclass(frozen=True)
class Event:
    title: str
    when: datetime

    def __post_init__(self):
        if not isinstance(self.when, datetime) or self.when.utcoffset() is None:
            raise ValueError("when must be a timezone-aware datetime")

    def __format__(self, spec):
        if not spec or spec == "local":
            return f"{self.title}: {self.when.isoformat(timespec='seconds')}"
        if spec == "utc":
            utc_value = self.when.astimezone(timezone.utc)
            return f"{self.title}: {utc_value.isoformat(timespec='seconds')}"
        if spec.startswith("date:"):
            directive = spec[len("date:"):]
            if not directive:
                raise ValueError("date: requires a strftime directive")
            return f"{self.title}: {format(self.when, directive)}"
        raise ValueError(f"unknown event mode: {spec!r}")


event = Event(
    "Release",
    datetime(2025, 1, 1, 0, 30, tzinfo=timezone(timedelta(hours=2))),
)
assert format(event) == "Release: 2025-01-01T00:30:00+02:00"
assert format(event, "utc") == "Release: 2024-12-31T22:30:00+00:00"
assert format(event, "date:%Y-%m-%d") == "Release: 2025-01-01"
raises(ValueError, lambda: Event("bad", datetime(2025, 1, 1)))
raises(ValueError, lambda: format(event, "date:"))
raises(ValueError, lambda: format(event, "tomorrow"))
print("Problem 5: PASS")

Problem 5: PASS


# Problem 6 — geographic coordinates with rounding/carry

**Difficulty:** advanced. Design `Coordinate(latitude, longitude)` with `decimal` (default), `decimal:N` (N decimal places, 0–6), and `dms` (degrees/minutes/seconds with N/S/E/W). Validate finite numbers and latitude/longitude bounds. For `dms`, round **total arcseconds first** and then use integer division; this correctly carries `59.999...` seconds into the next minute or degree. Use explicit hemisphere letters rather than relying on minus signs. The zero coordinate is N/E by convention for this exercise.

Expected: `format(Coordinate(40.5, -73.25), "dms") == '40°30\'00"N, 73°15\'00"W'`.

In [10]:
# SOLUTION 6
@dataclass(frozen=True)
class Coordinate:
    latitude: float
    longitude: float

    def __post_init__(self):
        for value, limit, name in (
            (self.latitude, 90, "latitude"),
            (self.longitude, 180, "longitude"),
        ):
            if not isinstance(value, (int, float)) or not math.isfinite(value):
                raise ValueError(f"{name} must be finite and numeric")
            if not -limit <= value <= limit:
                raise ValueError(f"{name} outside [-{limit}, {limit}]")

    @staticmethod
    def _dms(value, positive, negative):
        hemisphere = negative if value < 0 else positive
        rounded_seconds = round(abs(value) * 3_600)
        degrees, remainder = divmod(rounded_seconds, 3_600)
        minutes, seconds = divmod(remainder, 60)
        return f'{degrees}°{minutes:02d}\'{seconds:02d}"{hemisphere}'

    def __format__(self, spec):
        mode = spec or "decimal"
        if mode == "dms":
            return (f"{self._dms(self.latitude, 'N', 'S')}, "
                    f"{self._dms(self.longitude, 'E', 'W')}")
        if mode == "decimal":
            digits = 4
        elif re.fullmatch(r"decimal:[0-6]", mode):
            digits = int(mode[-1])
        else:
            raise ValueError(f"invalid coordinate spec: {spec!r}")
        return f"{self.latitude:.{digits}f}, {self.longitude:.{digits}f}"


nyc = Coordinate(40.5, -73.25)
assert format(nyc) == "40.5000, -73.2500"
assert format(nyc, "decimal:2") == "40.50, -73.25"
assert format(nyc, "dms") == '40°30\'00"N, 73°15\'00"W'
assert format(Coordinate(12.999999, 0), "dms").startswith('13°00\'00"N')
assert format(Coordinate(0, 0), "dms") == '0°00\'00"N, 0°00\'00"E'
raises(ValueError, lambda: Coordinate(float("nan"), 0))
raises(ValueError, lambda: Coordinate(91, 0))
raises(ValueError, lambda: format(nyc, "decimal:200"))
print("Problem 6: PASS")

Problem 6: PASS


# Problem 7 — nested formatting and an invoice row

**Difficulty:** advanced. Make `InvoiceLine(description, quantity, unit_price)` with strictly positive integer quantity and `Money` price. The `row:WIDTH` format spec (WIDTH 8–30) left-aligns the description in WIDTH characters, right-aligns the quantity in 3 characters, and right-aligns the total in 14 characters, separated by ` | `. If the description is too long, reject it rather than silently truncating. Empty spec returns a readable summary. The total must be computed with `Decimal`; formatting `Money` must reuse its own formatter, not duplicate currency logic.

*Note:* Python's width is generally code-point based, **not terminal display-column width**; East Asian wide glyphs and combining sequences need separate display-width tooling for exact terminal alignment.

In [11]:
# SOLUTION 7
@dataclass(frozen=True)
class InvoiceLine:
    description: str
    quantity: int
    unit_price: Money

    def __post_init__(self):
        if not isinstance(self.quantity, int) or isinstance(self.quantity, bool) or self.quantity <= 0:
            raise ValueError("quantity must be a positive integer")
        if not isinstance(self.unit_price, Money):
            raise TypeError("unit_price must be Money")
        if not isinstance(self.description, str) or not self.description:
            raise ValueError("description must be nonempty")

    @property
    def total(self):
        return Money(self.unit_price.amount * self.quantity, self.unit_price.currency)

    def __format__(self, spec):
        if not spec:
            return f"{self.quantity} × {self.description}: {format(self.total)}"
        match = re.fullmatch(r"row:(\d{1,2})", spec)
        if not match:
            raise ValueError(f"invalid invoice format: {spec!r}")
        width = int(match.group(1))
        if not 8 <= width <= 30:
            raise ValueError("description width must be 8..30")
        if len(self.description) > width:
            raise ValueError("description exceeds allocated width")
        price = format(self.total, "symbol|,.2f")
        if len(price) > 14:
            raise ValueError("price exceeds allocated width")
        return f"{self.description:<{width}} | {self.quantity:>3} | {price:>14}"


line = InvoiceLine("Notebook", 3, Money(Decimal("12.50")))
assert format(line) == "3 × Notebook: $37.50"
assert format(line, "row:10") == "Notebook   |   3 |         $37.50"
assert line.total.amount == Decimal("37.50")
raises(ValueError, lambda: format(line, "row:7"))
raises(ValueError, lambda: format(line, "row:abc"))
raises(ValueError, lambda: format(InvoiceLine("Extra long description", 1, positive), "row:8"))
raises(ValueError, lambda: InvoiceLine("Test", True, positive))
print("Problem 7: PASS")

Problem 7: PASS


# Problem 8 — inheritance and cooperative formatting

**Difficulty:** advanced. Define a `Record` base class whose empty spec returns its `str` form and whose `id` spec returns a stable record ID. Extend it with `TaggedRecord` that adds a `tag` spec; all other specs must be forwarded to the base class using `super().__format__(spec)`. Unsupported specs must raise `ValueError` rather than fall back to the object's identity string. Demonstrate again that `!s` bypasses the custom `id` grammar.

In [12]:
# SOLUTION 8
class Record:
    def __init__(self, record_id):
        if not isinstance(record_id, str) or not record_id:
            raise ValueError("record_id must be a nonempty string")
        self.record_id = record_id

    def __str__(self):
        return f"Record({self.record_id})"

    def __format__(self, spec):
        if not spec:
            return str(self)
        if spec == "id":
            return self.record_id
        raise ValueError(f"unknown record spec: {spec!r}")


class TaggedRecord(Record):
    def __init__(self, record_id, tag):
        super().__init__(record_id)
        self.tag = tag

    def __format__(self, spec):
        if spec == "tag":
            return f"{self.record_id} [{self.tag}]"
        return super().__format__(spec)


record = TaggedRecord("A-17", "ready")
assert format(record) == "Record(A-17)"
assert format(record, "id") == "A-17"
assert format(record, "tag") == "A-17 [ready]"
assert f"{record!s:>16}" == format(str(record), ">16")
raises(ValueError, lambda: format(record, "debug"))
print("Problem 8: PASS")

Problem 8: PASS


# Problem 9 — conversions, f-strings, and the exact format spec

**Difficulty:** advanced debugging. Implement a tracing formatter that records **only** method invocations, not prints. Pass a nested width/precision specification to `__format__`, verify which methods run for `!s` and `!r`, and show that `format(obj, "text")` cannot interpret `"text"` as a Python expression. Use assertions for an exact dispatch trace.

In [13]:
# SOLUTION 9
class TraceValue:
    def __init__(self, number):
        self.number = Decimal(number)
        self.log = []

    def __str__(self):
        self.log.append("str")
        return "text"

    def __repr__(self):
        self.log.append("repr")
        return "TraceValue"

    def __format__(self, spec):
        self.log.append(("format", spec))
        return format(self.number, spec)


traced = TraceValue("3.14159")
width, precision = 9, 3
assert f"{traced:{width}.{precision}f}" == format(Decimal("3.14159"), "9.3f")
assert f"{traced!s:>8}" == "    text"
assert f"{traced!r:<12}" == "TraceValue  "
assert traced.log == [("format", "9.3f"), "str", "repr"]
raises(ValueError, lambda: format(traced, "text"))
print("Problem 9: PASS", traced.log)

Problem 9: PASS [('format', '9.3f'), 'str', 'repr', ('format', 'text')]


# Problem 10 — defend against untrusted format specs

**Difficulty:** advanced engineering. An arbitrary field width can allocate an enormous string. Build `SafeText(text)` supporting ONLY the grammar `[ALIGN]WIDTH[.PRECISION]`, where ALIGN is optional `<`, `>`, or `^`, WIDTH is 1–40, PRECISION is 0–20, and absent precision means keep all characters. Reject all other fill characters, conversion flags, extra trailing tokens, huge integers, and empty width. An empty spec returns the raw string. Validate the *entire* input with `re.fullmatch` **before** delegating to `str.__format__`. For testing, use large-looking specs, never actually allocate huge output.

In [14]:
# SOLUTION 10
_SAFE_TEXT_SPEC = re.compile(r"(?P<align>[<>^]?)(?P<width>\d{1,3})(?:\.(?P<precision>\d{1,2}))?")


@dataclass(frozen=True)
class SafeText:
    text: str

    def __format__(self, spec):
        if not spec:
            return self.text
        match = _SAFE_TEXT_SPEC.fullmatch(spec)
        if match is None:
            raise ValueError("expected [<|>|^]WIDTH[.PRECISION]")
        width = int(match.group("width"))
        precision_text = match.group("precision")
        precision = int(precision_text) if precision_text is not None else None
        if not 1 <= width <= 40:
            raise ValueError("width must be 1..40")
        if precision is not None and precision > 20:
            raise ValueError("precision must be 0..20")
        # Only the validated, bounded spec reaches str.__format__.
        return format(self.text, spec)


safe = SafeText("Python")
assert format(safe) == "Python"
assert format(safe, "^10") == "  Python  "
assert format(safe, ">8.3") == "     Pyt"
assert format(safe, "<4.0") == "    "
for malicious in (">1000000000", "*^20", "^41", "<8.21", "4x", "", "999"):
    if malicious:
        raises(ValueError, lambda bad=malicious: format(safe, bad))
raises(ValueError, lambda: format(safe, "0"))
print("Problem 10: PASS")

Problem 10: PASS


# Problem 11 — units, normalization, and Decimal delegation

**Difficulty:** advanced. Store length canonically in meters as `Decimal`. Format grammar: `UNIT|NUMERIC_SPEC`, where UNIT is `m`, `cm`, or `km`. The default is `m|,.2f`. A unit without a numeric spec also uses `,.2f`. Reject invalid units and non-finite/negative lengths. Convert with *exact decimal factors*; then delegate to `Decimal.__format__`. A numeric spec can include alignment and grouping, and it is applied to the **number before the unit suffix**.

Example: `Length(Decimal('1234.5'))` in kilometers with `.4f` renders `1.2345 km`. For a truly untrusted numeric spec, add the bounded parser from Problem 10 rather than exposing all built-in formatting capabilities.

In [15]:
# SOLUTION 11
_LENGTH_FACTORS = {"m": Decimal("1"), "cm": Decimal("100"), "km": Decimal("0.001")}


@dataclass(frozen=True)
class Length:
    meters: Decimal

    def __post_init__(self):
        if not isinstance(self.meters, Decimal):
            raise TypeError("meters must be Decimal")
        if not self.meters.is_finite() or self.meters < 0:
            raise ValueError("meters must be finite and nonnegative")

    def __format__(self, spec):
        if not spec:
            unit, numeric_spec = "m", ",.2f"
        else:
            unit, sep, numeric_spec = spec.partition("|")
            if not sep or not numeric_spec:
                numeric_spec = ",.2f"
        if unit not in _LENGTH_FACTORS:
            raise ValueError(f"unknown unit: {unit!r}")
        converted = self.meters * _LENGTH_FACTORS[unit]
        return f"{format(converted, numeric_spec)} {unit}"


length = Length(Decimal("1234.5"))
assert format(length) == "1,234.50 m"
assert format(length, "km|.4f") == "1.2345 km"
assert format(Length(Decimal("1.25")), "cm") == "125.00 cm"
assert format(length, "m|>12,.1f") == "     1,234.5 m"
raises(ValueError, lambda: format(length, "mile|.2f"))
raises(ValueError, lambda: Length(Decimal("Infinity")))
raises(ValueError, lambda: Length(Decimal("-1")))
print("Problem 11: PASS")

Problem 11: PASS


# Problem 12 — capstone: a reusable report with true CSV escaping

**Difficulty:** expert. Create `InvoiceReport(lines)` that accepts a nonempty tuple of `InvoiceLine` objects in the *same currency*. Support `summary` (the default), `table:WIDTH` (width 8–30, rows delegated to `InvoiceLine.__format__`), and `csv` (header + rows + TOTAL). Use Python's `csv.writer` for correct commas/quotes rather than hand-joining CSV fields. Explicitly reject newline-containing descriptions in this example's one-row-per-item table contract; the CSV writer itself can represent newlines, but these are excluded here for parity with the table output. Totals and currency are computed with `Decimal`/`Money`, and the currency code is included in CSV numeric totals to prevent ambiguity.

**Testing:** verify proper CSV round-trip with `csv.reader`, correct total arithmetic, invalid modes, mixed currencies, and a too-long table description.

In [16]:
# SOLUTION 12
@dataclass(frozen=True)
class InvoiceReport:
    lines: tuple[InvoiceLine, ...]

    def __post_init__(self):
        if not isinstance(self.lines, tuple) or not self.lines:
            raise ValueError("lines must be a nonempty tuple")
        if any(not isinstance(line, InvoiceLine) for line in self.lines):
            raise TypeError("every item must be InvoiceLine")
        if len({line.unit_price.currency for line in self.lines}) != 1:
            raise ValueError("mixed-currency aggregation requires exchange-rate rules")
        if any("\n" in line.description or "\r" in line.description for line in self.lines):
            raise ValueError("descriptions must occupy one line")

    @property
    def total(self):
        currency = self.lines[0].unit_price.currency
        total = sum((line.total.amount for line in self.lines), Decimal("0"))
        return Money(total, currency)

    def __format__(self, spec):
        mode = spec or "summary"
        if mode == "summary":
            return f"{len(self.lines)} line(s) — total {format(self.total)}"
        if mode.startswith("table:"):
            match = re.fullmatch(r"table:(\d{1,2})", mode)
            if match is None:
                raise ValueError("expected table:WIDTH")
            width = int(match.group(1))
            if not 8 <= width <= 30:
                raise ValueError("table width must be 8..30")
            rows = [format(line, f"row:{width}") for line in self.lines]
            return "\n".join(rows + [f"TOTAL: {format(self.total)}"])
        if mode == "csv":
            buffer = StringIO(newline="")
            writer = csv.writer(buffer, lineterminator="\n")
            writer.writerow(("description", "quantity", "amount", "currency"))
            for line in self.lines:
                writer.writerow((line.description, line.quantity,
                                 format(line.total.amount, ".2f"),
                                 line.unit_price.currency))
            writer.writerow(("TOTAL", "", format(self.total.amount, ".2f"),
                             self.total.currency))
            return buffer.getvalue()
        raise ValueError(f"unknown report spec: {spec!r}")


report = InvoiceReport((
    InvoiceLine('Notebook, "A5"', 2, Money(Decimal("12.50"))),
    InvoiceLine("Pen", 3, Money(Decimal("1.20"))),
))
assert report.total.amount == Decimal("28.60")
assert format(report) == "2 line(s) — total $28.60"
parsed_rows = list(csv.reader(StringIO(format(report, "csv"))))
assert parsed_rows[0] == ["description", "quantity", "amount", "currency"]
assert parsed_rows[1] == ['Notebook, "A5"', "2", "25.00", "USD"]
assert parsed_rows[2] == ["Pen", "3", "3.60", "USD"]
assert parsed_rows[3] == ["TOTAL", "", "28.60", "USD"]
assert "TOTAL: $28.60" in format(report, "table:18")
raises(ValueError, lambda: format(report, "json"))
raises(ValueError, lambda: format(report, "table:100"))
raises(ValueError, lambda: format(report, "table:8"))
raises(ValueError, lambda: InvoiceReport((
    line, InvoiceLine("Euro", 1, Money(Decimal("1"), "EUR"))
)))
raises(ValueError, lambda: InvoiceReport((
    InvoiceLine("bad\nline", 1, positive),
)))
print("Problem 12: PASS")
print("CSV preview:\n" + format(report, "csv"))

Problem 12: PASS
CSV preview:
description,quantity,amount,currency
"Notebook, ""A5""",2,25.00,USD
Pen,3,3.60,USD
TOTAL,,28.60,USD



## Further challenges (deliberately left for independent practice)

1. Extend `Duration` with microsecond precision `clock.0` … `clock.6` without converting to floats; test negative values near zero.
2. Introduce an explicit `Money` rounding policy and decide whether quantization happens on input, per line item, or on the invoice grand total. Add tests for `Decimal("0.005")` and explain the business implications.
3. Implement width limits for *numeric* format specs, including fill/alignment, sign, grouping, precision and type, without blocking valid specs accidentally. Build a table of accepted/rejected strings.
4. Support grapheme-aware display columns using a suitable external Unicode display-width library (optional dependency); explain why ordinary `len` is insufficient.
5. Add an `html` report format with proper HTML escaping and separately test special characters, never treating user-controlled descriptions as markup.

**Review checklist:** Is the default spec documented? Are all unsupported specs rejected? Are the method return types strings? Is formatting deterministic and free of side effects? Are conversions and nested specs tested? Is delegated formatting used rather than reimplementing built-in grammars? Are expensive widths and precision bounded when specs come from untrusted input?

**Completion:** if the notebook runs from a fresh kernel without assertion failures, all 12 implemented problems and their regression tests passed.